# Datathon mobility data: retained-bundle review

## tl;dr
The retained bundles add **344,541 completed-status taxi records**, **168,959 airport movement records**, and **17,462,368 usable vessel GPS reports** as historical context. The checks below reconcile aggregate partitions and source receipts without needing the deleted downloads.

The app can explore recorded taxi corridors, airport activity by source-clock hour, and anonymous near-island device presence. These observations do **not** measure bus demand, unmet ferry demand, congestion or live vehicle positions.


## Context & Methods
This is a lightweight audit companion, not another ingestion test suite. It reads only the three retained `data/public/datathon/*.json` artifacts, uses Python's standard library, and can be rerun from the repository or this notebook's directory. No network, database or original ZIP files are required.

### Key assumptions
- `payment_completed` and `ride_completed` are treated as completed-status taxi records; the source has no status dictionary or unique journey IDs.
- Airport arrivals and departures keep their own event dates and times. Regions may be summed within a chosen year and event; aggregate medians and percentiles cannot be pooled.
- GPS observations use the source measurement clock and one unambiguous mapped island within approximately 500 m. Devices recur across days and islands.
- All source clock timezones are undocumented. No UTC conversion or cross-source timestamp join is attempted.

The original preparation scripts validate the pinned archives. This notebook independently checks the retained artifacts' internal consistency; it cannot independently reconstruct raw observations from aggregates.


## Data
### 1. Load the retained artifacts and record fingerprints


In [1]:
from collections import Counter
from hashlib import sha256
import json
from pathlib import Path

root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "data/public/datathon/airport.json").is_file()
)
bundles = {}
for name in ("taxis", "airport", "vessels"):
    artifact = root / "data/public/datathon" / f"{name}.json"
    content = artifact.read_bytes()
    bundles[name] = json.loads(content)
    print(f"{artifact.relative_to(root)} | {len(content):,} bytes")
    print(f"  SHA256 {sha256(content).hexdigest()}")
    for source in bundles[name]["sources"]:
        print(f"  Source: {source['url']}")


data\public\datathon\taxis.json | 74,543 bytes
  SHA256 3ce3d2175c2e7c39a613b90facf07e37c9c43cf1efe25c9207cdb434df474c99
  Source: https://github.com/mbs-datathon/maldives-datathon-2026-datasets/releases/download/v1.0/Avas_Ride.zip
data\public\datathon\airport.json | 1,120,535 bytes
  SHA256 8f2efefb0b181708c608c292f9813c56345ced29dc0bd90289aeaf5600e7c9e0
  Source: https://github.com/mbs-datathon/maldives-datathon-2026-datasets/releases/download/v1.0/MACL.zip
data\public\datathon\vessels.json | 1,674,256 bytes
  SHA256 a50d03c2e2ad6899ec41e349048809f06854424ed960fde81360d0b8d2cc5356
  Source: https://github.com/mbs-datathon/maldives-datathon-2026-datasets/releases/download/v1.0/FollowMe.zip


## Results
### 2. Reconcile taxi counts and mapped-corridor coverage
Daily and hourly arrays partition the same usable completed-status records. Corridor arrays include only journeys with both endpoints inside mapped island polygons; small groups may be withheld. Aggregate medians and percentiles are inspected as published, never summed or averaged.


In [2]:
taxi = bundles["taxis"]
coverage = taxi["coverage"]
completed = coverage["completed_records"]
assert coverage["source_rows"] == sum(coverage["status_counts"].values())
assert taxi["summary"]["ride_count"] == completed
assert sum(row["ride_count"] for row in taxi["daily"]) == completed
assert sum(row["ride_count"] for row in taxi["hourly"]) == completed
assert len({row["date"] for row in taxi["daily"]}) == coverage["observed_days"]
assert len({row["hour"] for row in taxi["hourly"]}) == len(taxi["hourly"])
minimum_group = taxi["quality"]["minimum_profile_records"]
for name in ("daily", "hourly", "zones", "zoneHourly", "corridors", "corridorHourly"):
    assert all(row["ride_count"] >= minimum_group for row in taxi[name])
    assert all(0 < row["median_minutes"] <= row["p90_minutes"] for row in taxi[name])
matched = coverage["both_endpoints_matched"]
corridor_records = sum(row["ride_count"] for row in taxi["corridors"])
hourly_corridor_records = sum(row["ride_count"] for row in taxi["corridorHourly"])
assert hourly_corridor_records <= corridor_records <= matched <= completed
print(f"Source rows: {coverage['source_rows']:,}")
print(f"Usable completed-status records: {completed:,}")
print(f"Hourly and daily totals each reconcile: {completed:,}")
print(f"Both endpoints mapped: {matched:,} ({matched / completed:.1%} of usable records)")
print(f"Retained corridor records: {corridor_records:,}")
print(f"Retained corridor-hour records: {hourly_corridor_records:,}")
print(f"Expired-status records excluded from trip metrics: {taxi['summary']['expired_records']:,}")
print(f"Observed source dates: {coverage['date_from']} to {coverage['date_to']}")
print("PASS: counts, profile thresholds and percentile ordering")


Source rows: 371,788
Usable completed-status records: 344,541
Hourly and daily totals each reconcile: 344,541
Both endpoints mapped: 303,990 (88.2% of usable records)
Retained corridor records: 303,990
Retained corridor-hour records: 303,473
Expired-status records excluded from trip metrics: 27,247
Observed source dates: 2026-08-01 to 2026-09-30
PASS: counts, profile thresholds and percentile ordering


### 3. Reconcile airport event counts and passenger cohorts
Monthly aggregates include valid event dates even when a source time is missing. Hourly aggregates additionally require a valid actual time. Epoch sentinel and outside-period arrival dates are excluded rather than borrowing a departure date. Quality flags overlap: the 1970 sentinel count is a subset of outside-period dates.


In [3]:
airport = bundles["airport"]
measures = ("movement_records", "passengers", "adults", "children", "infants")
for level in ("monthly", "hourly"):
    rows = airport[level]
    keys = ("period", "event", "region") if level == "monthly" else ("year", "hour", "event", "region")
    assert len({tuple(row[key] for key in keys) for row in rows}) == len(rows)
    assert all(row["passengers"] == row["adults"] + row["children"] + row["infants"] for row in rows)
    assert all(row[measure] >= 0 for row in rows for measure in measures)
    if level == "hourly":
        assert all(0 <= row["hour"] <= 23 for row in rows)
    for event in ("arrival", "departure"):
        selected = [row for row in rows if row["event"] == event]
        expected = airport["coverage"]["eventTotals"][f"{event}_{level}"]
        for measure in measures:
            assert sum(row[measure] for row in selected) == expected[measure]
counts = airport["quality"]["counts"]
for event in ("arrival", "departure"):
    raw = airport["quality"]["rawEventTotals"][event]
    monthly = airport["coverage"]["eventTotals"][f"{event}_monthly"]
    hourly = airport["coverage"]["eventTotals"][f"{event}_hourly"]
    assert raw["passengers"] == monthly["passengers"] + counts.get(f"{event}_excluded_date_passengers", 0)
    assert monthly["passengers"] == hourly["passengers"] + counts.get(f"{event}_excluded_hour_passengers", 0)
    assert raw["movement_records"] == airport["coverage"]["records"]
    print(f"{event.title()}: {monthly['movement_records']:,} monthly / {hourly['movement_records']:,} hourly events")
    print(f"  Passengers: {monthly['passengers']:,} monthly / {hourly['passengers']:,} hourly")
    print(f"  Date-excluded records: {raw['movement_records'] - monthly['movement_records']:,}; additional time-excluded records: {monthly['movement_records'] - hourly['movement_records']:,}")
assert airport["quality"]["unique_record_ids"] == airport["coverage"]["records"]
print(f"1970 arrival sentinels: {counts['arrival_epoch_sentinel']}; repeated non-ID content retained: {counts['repeated_content_excluding_uid_retained']}")
print("PASS: unique grain, passenger cohorts, event counts and exclusion reconciliation")


Arrival: 168,946 monthly / 168,859 hourly events
  Passengers: 25,686,620 monthly / 25,673,561 hourly
  Date-excluded records: 13; additional time-excluded records: 87
Departure: 168,959 monthly / 168,873 hourly events
  Passengers: 25,602,263 monthly / 25,589,764 hourly
  Date-excluded records: 0; additional time-excluded records: 86
1970 arrival sentinels: 11; repeated non-ID content retained: 5
PASS: unique grain, passenger cohorts, event counts and exclusion reconciliation


### 4. Pool only the selected airport year and event
This reproduces the panel's default **2025 arrivals** profile by summing additive passenger counts across regions. Absent hours remain absent. The peak is a cumulative annual source-clock bin, not a claim about passengers arriving on an average day. Arrival and departure events and other years are not mixed.


In [4]:
selected_year = 2025
selected_event = "arrival"
selected_rows = [
    row for row in airport["hourly"]
    if row["year"] == selected_year and row["event"] == selected_event
]
passengers_by_hour = Counter()
records_by_hour = Counter()
for row in selected_rows:
    passengers_by_hour[row["hour"]] += row["passengers"]
    records_by_hour[row["hour"]] += row["movement_records"]
assert sum(passengers_by_hour.values()) == sum(row["passengers"] for row in selected_rows)
peak_hour = max(passengers_by_hour, key=passengers_by_hour.get)
print(f"Default profile: {selected_year} {selected_event}s")
print(f"Observed hour bins: {len(passengers_by_hour)}; no absent bin is filled with zero")
print(f"Timed passenger movements: {sum(passengers_by_hour.values()):,}")
print(f"Peak source-clock bin: {peak_hour:02}:00, {passengers_by_hour[peak_hour]:,} cumulative passenger movements")
print("Top three observed hour bins:")
for hour, passengers in passengers_by_hour.most_common(3):
    print(f"  {hour:02}:00 | {passengers:,} passengers | {records_by_hour[hour]:,} movement records")
for event in ("arrival", "departure"):
    annual = sum(row["passengers"] for row in airport["monthly"] if row["year"] == selected_year and row["event"] == event)
    print(f"{selected_year} {event} passenger movements, valid-date monthly rows: {annual:,}")
print("PASS: year/event filter and additive regional pooling")


Default profile: 2025 arrivals
Observed hour bins: 24; no absent bin is filled with zero
Timed passenger movements: 2,701,912
Peak source-clock bin: 11:00, 352,130 cumulative passenger movements
Top three observed hour bins:
  11:00 | 352,130 passengers | 2,273 movement records
  09:00 | 340,059 passengers | 1,473 movement records
  07:00 | 278,994 passengers | 1,323 movement records
2025 arrival passenger movements, valid-date monthly rows: 2,702,086
2025 departure passenger movements, valid-date monthly rows: 2,673,014
PASS: year/event filter and additive regional pooling


### 5. Reconcile vessel GPS partitions and near-island records
GPS-report counts are additive across the accepted spatial partition. Unique device counts are not additive across dates or islands. A near-island observation does not establish a ferry sailing, harbour arrival or passenger service.


In [5]:
vessels = bundles["vessels"]
coverage = vessels["coverage"]
partition = ("matched_nearshore_records", "ambiguous_nearshore_records", "outside_nearshore_records")
assert sum(coverage[key] for key in partition) == coverage["retained_records"]
assert sum(row["gps_records"] for row in vessels["daily"]) == coverage["retained_records"]
assert sum(row["gps_records"] for row in vessels["islands"]) == coverage["matched_nearshore_records"]
assert len(vessels["file_receipts"]) == coverage["files"] == 31
assert sum(row["rows"] for row in vessels["file_receipts"]) == coverage["rows"]
assert len({row["island_id"] for row in vessels["islands"]}) == coverage["islands_with_records"]
assert len({row["date"] for row in vessels["daily"]}) == coverage["days"] == 31
for island in vessels["islands"]:
    assert sum(day["gps_records"] for day in island["daily"]) == island["gps_records"]
    assert len(island["daily"]) == island["days_observed"]
    assert 0 < island["tracked_devices"] <= coverage["tracked_devices"]
print(f"Source GPS reports: {coverage['rows']:,}")
print(f"Retained reports: {coverage['retained_records']:,}")
for key in partition:
    print(f"  {key}: {coverage[key]:,} ({coverage[key] / coverage['retained_records']:.1%})")
print(f"Tracked devices across source period: {coverage['tracked_devices']:,}; not a national fleet census")
print(f"Islands with unambiguous recorded presence: {coverage['islands_with_records']:,}")
print("PASS: daily, file, spatial-partition and per-island report reconciliation")


Source GPS reports: 17,462,749
Retained reports: 17,462,368
  matched_nearshore_records: 5,560,881 (31.8%)
  ambiguous_nearshore_records: 2,497,669 (14.3%)
  outside_nearshore_records: 9,403,818 (53.9%)
Tracked devices across source period: 1,776; not a national fleet census
Islands with unambiguous recorded presence: 1,221
PASS: daily, file, spatial-partition and per-island report reconciliation


## Takeaways
- Taxi hourly and daily partitions reconcile to **344,541 usable completed-status records**. Only **303,990** have both endpoints matched to the retained island polygons; corridor coverage should stay visible when interpreting rankings.
- The default airport profile pools additive passenger counts across regions for **2025 arrivals**, keeping events and years separate. The peak source-clock bin is **11:00** with **352,130 cumulative 2025 arriving passenger movements**. It is not an average hourly arrival rate.
- Vessel spatial partitions reconcile to **17,462,368 retained GPS reports**. Ambiguous and offshore reports remain excluded from island presence. Device totals must not be summed as a unique fleet.
- Source clocks, dates, populations and coverage differ. These sources support historical investigation, not a combined bus-demand model or a replacement for published ferry services.
- Taxi medians, percentiles and recorded speed proxies must not be summed or averaged to invent pooled journey statistics. Compare their original groups or recompute from raw records when an authorized future analysis needs a true pooled metric.
- MACL observations remain separate from CAA/MBS totals. Anonymous route codes cannot substantiate a named island connection; transit counts are not added to arriving/departing passenger cohorts.

Original sources: the [official Datathon release v1.0](https://github.com/mbs-datathon/maldives-datathon-2026-datasets/releases/tag/v1.0). Each retained JSON preserves its archive/member provenance and additional interpretation limits.


In [6]:
print("All retained-bundle review checks completed successfully.")
print("Raw archive equality is established by the preparation receipts, not reverified by this notebook.")


All retained-bundle review checks completed successfully.
Raw archive equality is established by the preparation receipts, not reverified by this notebook.
